In [8]:
import pyrosetta
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector, NeighborhoodResidueSelector, OrResidueSelector
from pyrosetta.rosetta.core.select.movemap import MoveMapFactory, move_map_action
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

# Initialize PyRosetta (Heavy-hitting flags enabled for the whole session)
pyrosetta.init("-mute all -ex1 -ex2 -use_input_sc")

# 1. Load the AlphaFold PDB
# UPDATE THIS PATH TO YOUR ALPHAFOLD PDB!
af_pdb_path = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/clean_complex_bad_good_spot.pdb"
binder_chain = "G"
target_chains = "A,B,C,D,E,F"

print(f"Loading AlphaFold model...")
# Replace the old pose_from_pdb line with this:
alphafold_pose = pyrosetta.pose_from_pdb(af_pdb_path)
scorefxn = pyrosetta.get_fa_scorefxn()

# 2. Select the interface (10.0 Å bubble to allow clash resolution)
binder_sel = ChainSelector(binder_chain)
move_interface_sel = NeighborhoodResidueSelector(binder_sel, 10.0, True)
move_zone = OrResidueSelector(binder_sel, move_interface_sel)

# 3. Set up the MoveMap (Allow backbones and sidechains to shift, no mutations)
mm_factory = MoveMapFactory()
mm_factory.all_bb(False)
mm_factory.all_chi(False)
mm_factory.all_jumps(False)
mm_factory.add_chi_action(move_map_action.mm_enable, move_zone)
mm_factory.add_bb_action(move_map_action.mm_enable, move_zone)
movemap = mm_factory.create_movemap_from_pose(alphafold_pose)

# 4. Run FastRelax
print("Settling AlphaFold geometry into Rosetta physics... (Grab a coffee)")
relax = FastRelax(scorefxn, 5)
relax.set_movemap(movemap)
relax.apply(alphafold_pose) # Modifies alphafold_pose directly in memory

# 5. Check the baseline ddG
iam = InterfaceAnalyzerMover(f"{binder_chain}_{target_chains.replace(',', '')}")
iam.set_scorefunction(scorefxn)
iam.set_pack_separated(True)
iam.apply(alphafold_pose)

baseline_ddg = iam.get_interface_dG()
baseline_bsa = iam.get_interface_delta_sasa()

print("\n========================================")
print(f"🌟 SETTLED BASELINE ddG: {baseline_ddg:.2f} REU")
print(f"🌟 SETTLED BASELINE BSA: {baseline_bsa:.2f} Å²")
print("========================================")
print("alphafold_pose is now stored in memory and ready for Cell 2!")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2026.06+release.1a56185c2592611dec4c9c75ddc9468cd2227c1f 2026-01-30T13:14:27] retrieved from: http://www.pyrosetta.org
Loading AlphaFold model...
Settling AlphaFold geometry into Rosetta physics... (Grab a coffee)

🌟 SETTLED BASELINE ddG: -6.5

# Backup for Alphafold
Synthesize a full pdb using pymol, then use MoveMap to quickly relax

In [11]:
# Assuming alphafold_pose is loaded in memory from Cell 1
import os
from pyrosetta.rosetta.core.select.residue_selector import AndResidueSelector, NotResidueSelector, ChainSelector, NeighborhoodResidueSelector, OrResidueSelector
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import OperateOnResidueSubset, PreventRepackingRLT, RestrictToRepackingRLT
from pyrosetta.rosetta.protocols.relax import FastRelax
import pyrosetta

# Define Output Directory
out_dir = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/decoy_run_1/"
os.makedirs(out_dir, exist_ok=True)

# 1. Selectors for Mutagenesis
target_sel = ChainSelector(target_chains)

# Tight 5.0 Å bubble for mutations
mut_interface_sel = NeighborhoodResidueSelector(binder_sel, 5.0, True)
target_interface = AndResidueSelector(target_sel, mut_interface_sel)
frozen_sel = NotResidueSelector(mut_interface_sel)

# Looser 8.0 Å bubble for backbone movement
move_interface_sel = NeighborhoodResidueSelector(binder_sel, 8.0, True)
move_zone = OrResidueSelector(binder_sel, move_interface_sel)

# 2. Task Factory (What mutates?)
tf = TaskFactory()
tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(), frozen_sel))
tf.push_back(OperateOnResidueSubset(RestrictToRepackingRLT(), target_interface))

# 3. Bulletproof MoveMap (What moves?)
movemap = pyrosetta.rosetta.core.kinematics.MoveMap()
movemap.set_bb(False)
movemap.set_chi(False)
movemap.set_jump(False)

# Apply selector to get a boolean vector (True for residues in the move zone)
move_vector = move_zone.apply(alphafold_pose)

# Loop through and manually enable movement for the selected residues
for i in range(1, alphafold_pose.total_residue() + 1):
    if move_vector[i]:
        movemap.set_bb(i, True)
        movemap.set_chi(i, True)

# 4. Set up FastRelax as a Designer
print("Setting up Sequence Designer...")
designer = FastRelax(scorefxn, 5) 
designer.set_task_factory(tf)
designer.set_movemap(movemap)

# 5. The Decoy Loop
num_decoys = 45 # Crank this up as high as your timeframe allows!
best_ddg = 0.0

print(f"Starting {num_decoys} design trajectories. Saving FASTA outputs to {out_dir}")

for i in range(num_decoys):
    print(f"\n--- Trajectory {i+1}/{num_decoys} ---")
    
    # Clone the clean baseline
    decoy_pose = alphafold_pose.clone()
    
    # Run Design
    designer.apply(decoy_pose)
    
    # Score
    iam.apply(decoy_pose)
    current_ddg = iam.get_interface_dG()
    current_bsa = iam.get_interface_delta_sasa()
    
    print(f"Result -> ddG: {current_ddg:.2f} REU | BSA: {current_bsa:.2f} Å²")
    
    # ----- HIGH-THROUGHPUT FASTA OUTPUT -----
    
    # Create an individual, score-labeled FASTA file for this trajectory
    fasta_filename = os.path.join(out_dir, f"decoy_{i+1}_ddG_{current_ddg:.2f}.fasta")
    
    with open(fasta_filename, "w") as f:
        # Write standard Multi-FASTA format for AlphaFold Multimer
        for chain_id in range(1, decoy_pose.num_chains() + 1):
            seq = decoy_pose.chain_sequence(chain_id)
            f.write(f">Chain_{chain_id}\n")
            f.write(f"{seq}\n")
    
    # Track the absolute best score just for terminal printing
    if current_ddg < best_ddg:
        best_ddg = current_ddg

print("\n========================================")
print(f"✅ PyRosetta run complete.")
print(f"✅ {num_decoys} individual FASTA files saved to: {out_dir}")
print(f"🏆 Best PyRosetta ddG recorded: {best_ddg:.2f}")
print("========================================")

Setting up Sequence Designer...
Starting 45 design trajectories. Saving FASTA outputs to /home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/decoy_run_1/

--- Trajectory 1/45 ---
Result -> ddG: -29.59 REU | BSA: 1620.91 Å²

--- Trajectory 2/45 ---
Result -> ddG: -28.60 REU | BSA: 1537.97 Å²

--- Trajectory 3/45 ---
Result -> ddG: -25.64 REU | BSA: 1362.43 Å²

--- Trajectory 4/45 ---
Result -> ddG: -32.00 REU | BSA: 1506.24 Å²

--- Trajectory 5/45 ---
Result -> ddG: -26.82 REU | BSA: 1333.10 Å²

--- Trajectory 6/45 ---
Result -> ddG: -33.04 REU | BSA: 1584.79 Å²

--- Trajectory 7/45 ---
Result -> ddG: -30.72 REU | BSA: 1563.56 Å²

--- Trajectory 8/45 ---
Result -> ddG: -34.08 REU | BSA: 1568.72 Å²

--- Trajectory 9/45 ---
Result -> ddG: -30.50 REU | BSA: 1695.44 Å²

--- Trajectory 10/45 ---
Result -> ddG: -39.18 REU | BSA: 1681.72 Å²

--- Trajectory 11/45 ---
Result -> ddG: -32.86 REU | BSA: 1697.70 Å²

--- Trajectory 12/45 ---
Result -> ddG: -25.85 REU | BSA: 1386.77 Å²

--- Traj

In [ ]:
import os

# Amino acid 3-to-1 letter dictionary
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

input_pdb = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/af_optimized_binder_best.pdb"

# Define output paths in the same directory
output_fasta = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/af_optimized_binder_best.fasta"
output_af3 = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/af3_multimer_sequence.txt"

sequences = {}
last_res_num = {}

# Parse the PDB file
with open(input_pdb, "r") as f:
    for line in f:
        # Look for Alpha-Carbons (CA)
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain_id = line[21]
            res_num = int(line[22:26].strip())
            res_name = line[17:20].strip()
            
            # Initialize chain if we haven't seen it
            if chain_id not in sequences:
                sequences[chain_id] = ""
                last_res_num[chain_id] = -9999
            
            # Add amino acid, avoiding duplicate alt-loc atoms
            if res_num != last_res_num[chain_id]:
                sequences[chain_id] += d3to1.get(res_name, "X")
                last_res_num[chain_id] = res_num

# Write Output 1: Standard Multi-FASTA
with open(output_fasta, "w") as f:
    for chain, seq in sequences.items():
        f.write(f">Chain_{chain}\n")
        f.write(f"{seq}\n")

# Write Output 2: ColabFold / AF3 Format (Colon-Separated)
colabfold_list = list(sequences.values())
with open(output_af3, "w") as f:
    f.write(":".join(colabfold_list))

print(f"✅ Multi-FASTA saved to: {output_fasta}")
print(f"✅ AlphaFold sequence saved to: {output_af3}")

✅ Multi-FASTA saved to: /home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/af_optimized_binder_best.fasta
✅ AlphaFold sequence saved to: /home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/af3_multimer_sequence.txt
